# Papers as recipes — run training-free FLUX methods as configs (or generate a pipeline)
Each published method here is **one recipe row**. This notebook shows the two ways to use them:
1. **as a config** — `RecipeRunner.run(recipe, inputs)` (explore, sweep, compose), and
2. **as a shipped pipeline** — `scripts/gen_pipeline.py <recipe>` codegens a `trust_remote_code` block, no
   hand-written denoise loop.
Plus a composition that *no* single paper wrote (`structure_appearance` = `freecontrol` ⊕ `appearance`).
Runtime: 80GB A100, FLUX.1-dev license.

In [ ]:
import subprocess, os
BRANCH = "main"
for _ in range(3):
    if subprocess.call(["pip","install","-q","git+https://github.com/huggingface/diffusers.git"])==0: break
!pip install -q transformers accelerate sentencepiece protobuf hf_transfer scikit-image pyyaml
subprocess.run(["rm","-rf","flux-recipes"])
subprocess.run(["git","clone","-q","-b",BRANCH,"https://github.com/remyxai/flux-recipes.git"])

In [ ]:
import sys, torch, numpy as np
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
sys.path.insert(0,"flux-recipes")
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
assert torch.cuda.is_available(); print("GPU:",torch.cuda.get_device_name(0))
from flux_modular import RecipeRunner, load_recipes
runner=RecipeRunner(steps=20); R=load_recipes("flux-recipes/recipes")

## The library — each row is a paper (or a composition of them)

In [ ]:
import pandas as pd
rows=[{"recipe":k, "run-loop":v.get("run","default"), "paper":v.get("paper","-"),
       "validated":v.get("validated","?").split("#")[0].strip(),
       "what":(v.get("description","")[:60])} for k,v in sorted(R.items())]
pd.set_option("display.max_colwidth", 64)
print(pd.DataFrame(rows).to_string(index=False))

## 1. Run a few papers as configs (one `runner.run` per method)

In [ ]:
from PIL import Image, ImageDraw
from skimage import data
from IPython.display import display
SRC=Image.fromarray(data.astronaut()).convert("RGB").resize((1024,1024))
APP=Image.fromarray(data.coffee()).convert("RGB").resize((1024,1024))
def grid(items,c=250):
    g=Image.new("RGB",(len(items)*c+(len(items)+1)*6,c+40),"white"); d=ImageDraw.Draw(g)
    for j,(n,im,sub) in enumerate(items):
        x=6+j*(c+6); g.paste(im.resize((c,c)),(x,4)); d.text((x+4,c+8),n[:30],fill="black")
        if sub: d.text((x+4,c+22),sub[:30],fill="black")
    display(g)
runs=[
 ("freecontrol\n2511.05219", "freecontrol", {"prompt":"a bronze statue bust","ref_structure":SRC}, {}),
 ("regional\n2302.08113", "regional", {"base_prompt":"a living room","regions":[{"prompt":"a red sofa","bbox":[0,0.5,0.5,1]},{"prompt":"a green plant","bbox":[0.6,0.2,1,1]}]}, {}),
 ("story\n2405.01434", "story", {"character_prompt":"a red-haired woman with freckles","scene_prompts":["in a cafe #a","in a forest #b"]}, {}),
 ("appearance\n2603.26767", "appearance", {"prompt":"a portrait of a person","ref_appearance":APP}, {}),
]
tiles=[]
for name,rec,inp,ov in runs:
    out=runner.run(rec, inp, seed=0, **ov); im=out[0] if isinstance(out,list) else out
    tiles.append((name.split(chr(92)+"n")[0], im, name.split(chr(92)+"n")[1]))
grid(tiles)
print("Four papers, four `runner.run(recipe, inputs)` calls — same interface, different config.")

## 2. A composition no paper wrote — `structure_appearance` = freecontrol ⊕ appearance

In [ ]:
comp=runner.run("structure_appearance", {"prompt":"a portrait of a person","ref_structure":SRC,"ref_appearance":APP}, S=0.5, redux_scale=1.5, seed=0)
grid([("ref_structure",SRC,"pose"),("ref_appearance",APP,"material"),("composition",comp,"structure ⊕ appearance")])
print("Structure from one image, appearance from another, content from the prompt — a new capability that is")
print("just a merged recipe, not new code.")

## 3. Ship a paper as a pipeline — codegen (the CLI)

In [ ]:
# recipe -> a standalone trust_remote_code pipeline dir (block.py + configs + vendored primitive), no denoise loop written
!cd flux-recipes && PYTHONPATH=. python scripts/gen_pipeline.py freecontrol /content/freecontrol-gen
print("\nGenerated files:"); print(sorted(os.listdir("/content/freecontrol-gen")))
from diffusers import ModularPipeline
p=ModularPipeline.from_pretrained("/content/freecontrol-gen", trust_remote_code=True)
print("loads as:", type(p.blocks).__name__, "-> upload this dir to the Hub and it's a one-line pipeline.")
print("head of the generated block.py:")
print("".join(open("/content/freecontrol-gen/block.py").readlines()[:22]))